Avoid calling the Lakehouse Federation / Lakebase token endpoint on every request. The fix in model serving is to fetch once per process (in load_context) and then reuse and refresh the token in predict, not mint a new one each time.

1. Move token fetch to load_context
load_context runs once per model instance when the serving container starts or scales up, so it’s the right place for the initial token acquisition. Here _get_token() does the OIDC POST; you call it once at startup instead of per request.

2. Refresh only when close to expiry
Add a small helper that checks expiry and refreshes if needed.

In [0]:
class FraudModelPOC(mlflow.pyfunc.PythonModel):
    TOKEN_SAFETY_MARGIN = 60  # seconds

    def _get_token(self) -> tuple[str, float]:
        data = {
            "grant_type": "client_credentials",
            "client_id": CLIENT_ID,
            "client_secret": CLIENT_SECRET,
            "scope": SCOPES,
        }
        headers = {"Content-Type": "application/x-www-form-urlencoded"}
        resp = requests.post(OIDC_TOKEN_URL, data=data, headers=headers, timeout=5)
        resp.raise_for_status()
        token = resp.json()["access_token"]
        # Example: if token TTL is 3600s, set expiry accordingly (you can parse `expires_in`)
        ttl = resp.json().get("expires_in", 3600)
        return token, time.time() + ttl

    def _ensure_token(self):
        now = time.time()
        if (
            getattr(self, "_token", None) is None
            or getattr(self, "_token_expiry", 0) - self.TOKEN_SAFETY_MARGIN < now
        ):
            self._token, self._token_expiry = self._get_token()

    def load_context(self, context):
        # Prime the token once at startup
        self._ensure_token()

    def predict(self, context, model_input: pd.DataFrame):
        # Cheap check, refresh only if needed
        self._ensure_token()
        oauth_token = self._token
        client_id = CLIENT_ID

        # use oauth_token in psycopg2.connect(...)
